In [1]:
import warnings

warnings.filterwarnings("ignore", message="Pydantic serializer warnings")

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "yellow"

In [4]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3:14b", temperature=0)

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    context_schema=ColourContext  
)

In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='5318da9a-c4ae-404d-a3c3-04f6b1df24fb'),
              AIMessage(content='I’m not sure—what’s your favourite colour?', additional_kwargs={}, response_metadata={'model': 'gpt-oss:20b', 'created_at': '2026-02-28T20:51:57.308702985Z', 'done': True, 'done_reason': 'stop', 'total_duration': 121304329982, 'load_duration': 74759292258, 'prompt_eval_count': 73, 'prompt_eval_duration': 20625931627, 'eval_count': 117, 'eval_duration': 25480431065, 'logprobs': None, 'model_name': 'gpt-oss:20b', 'model_provider': 'ollama'}, id='lc_run--019ca603-e47e-7e13-91ee-992cbbe9466e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 73, 'output_tokens': 117, 'total_tokens': 190})]}


In [8]:
print(response["messages"][-1].content)

I’m not sure—what’s your favourite colour?


## Accessing Context

In [9]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [10]:
agent = create_agent(
    model=model,
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

In [11]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='3b79dea5-b58d-49e7-aa17-9c536dae4456'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-oss:20b', 'created_at': '2026-02-28T20:52:12.822418617Z', 'done': True, 'done_reason': 'stop', 'total_duration': 16340669074, 'load_duration': 481790817, 'prompt_eval_count': 143, 'prompt_eval_duration': 8840999135, 'eval_count': 33, 'eval_duration': 6991346837, 'logprobs': None, 'model_name': 'gpt-oss:20b', 'model_provider': 'ollama'}, id='lc_run--019ca605-b388-7b30-9d1f-d053b6d4fb84-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': '8d2a62c4-cfd8-4edb-900b-271dd3afc8b0', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 143, 'output_tokens': 33, 'total_tokens': 176}),
              ToolMessage(content='blue', name='get_favourite_colour', id='42b43a88-04f0-4a20-a2e7-ef949821272d', tool_call_id='8d2a

In [12]:
print(response["messages"][-1].content)

Your favourite colour is **blue**.


In [13]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="green")
)

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='15d08baf-c574-4a01-8142-d9dab789e498'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-oss:20b', 'created_at': '2026-02-28T20:52:36.982659875Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7739427581, 'load_duration': 410716816, 'prompt_eval_count': 143, 'prompt_eval_duration': 205051694, 'eval_count': 33, 'eval_duration': 7102457923, 'logprobs': None, 'model_name': 'gpt-oss:20b', 'model_provider': 'ollama'}, id='lc_run--019ca606-2fb8-7a10-b12c-613f98550589-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': '5fca4ae6-9dce-43e1-92b5-d0fd207d2088', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 143, 'output_tokens': 33, 'total_tokens': 176}),
              ToolMessage(content='green', name='get_favourite_colour', id='5f6c5b9e-367b-48c3-b3f1-6c16c6a89ae8', tool_call_id='5fca4

In [14]:
print(response["messages"][-1].content)

Your favourite colour is **green**.
